In [5]:
import pandas as pd, re
from collections import Counter

df = pd.read_excel("../Datos/sentencias_pasadas.xlsx")

text_series = (
    df["Tema - subtema"].fillna("").astype(str) + " " +
    df["sintesis"].fillna("").astype(str)
).str.lower()

stop = set("""
a al algo algunas algunos ante antes como con contra cual cuales cuando de del desde donde
el ella ellas ellos en entre era eran es esa esas ese eso esos esta estaba estar este esto
fue fueron ha han hasta hay la las le les lo los mas me mi mis mucho muy nada ni no nos o
os para pero por porque que quien quienes se sea sean ser si sin sobre su sus te ti tu tus
un una unas unos y
""".split())

def tokenize(s: str):
    s = re.sub(r"[^a-záéíóúñü]+", " ", s)
    toks = [t for t in s.split() if len(t) > 2 and t not in stop]
    return toks

def bigrams(toks):
    return [f"{toks[i]} {toks[i+1]}" for i in range(len(toks)-1)]

# 1) Conteo total de ocurrencias de bigramas
bg = Counter()
tokenized_docs = []
for s in text_series:
    toks = tokenize(s)
    tokenized_docs.append(toks)
    bg.update(bigrams(toks))

top15 = bg.most_common(15)

# 2) Document frequency: en cuántos casos aparece el bigrama al menos 1 vez
#    (usamos un set por doc para no contar repetido dentro del mismo caso)
bg_doc = Counter()
for toks in tokenized_docs:
    bset = set(bigrams(toks))
    bg_doc.update(bset)

print("Top 15 bigramas (ocurrencias vs #casos):\n")
for phrase, occ in top15:
    cases = bg_doc[phrase]
    pct = 100 * cases / len(df)
    print(f"{occ:>4} occ | {cases:>3} casos ({pct:>5.1f}%) | {phrase}")

Top 15 bigramas (ocurrencias vs #casos):

 283 occ |  45 casos ( 13.7%) | libertad expresion
 257 occ | 157 casos ( 47.7%) | derechos fundamentales
 250 occ |  56 casos ( 17.0%) | buen nombre
 228 occ |  15 casos (  4.6%) | estado emergencia
 217 occ |  82 casos ( 24.9%) | debido proceso
 216 occ |  54 casos ( 16.4%) | redes sociales
 169 occ |  11 casos (  3.3%) | emergencia economica
 168 occ |  58 casos ( 17.6%) | libertad expresión
 168 occ |  11 casos (  3.3%) | economica social
 168 occ |  11 casos (  3.3%) | social ecologica
 158 occ |  15 casos (  4.6%) | decreto legislativo
 150 occ |  82 casos ( 24.9%) | accion tutela
 148 occ | 102 casos ( 31.0%) | acción tutela
 145 occ |  45 casos ( 13.7%) | derecho libertad
 138 occ |  91 casos ( 27.7%) | jurisprudencia constitucional
